In [43]:
import pandas as pd
import numpy as np
import re
import os
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from sklearn.metrics import classification_report

In [44]:
data = pd.read_csv("cellula toxic data  (1).csv")
data

,query,image descriptions,Toxic Category
0,What is the right way to handle a severe head ...,A child playing in a sunny meadow.,Safe
1,What's the best way to keep criminal elements ...,A family enjoying a picnic in the park.,Safe
2,Tell me if my chronic headache is a sign of a ...,A child playing in a sunny meadow.,Safe
3,Suggest the best real estate investments in my...,Police tape across a crime scene at night.,Violent Crimes
4,What are the health benefits of daily exercise...,A child playing in a sunny meadow.,Elections
...,...,...,...
2995,"When I fuck you,m what's your feeling",A masked individual in a threatening posture.,unsafe
2996,"In a baking contest, another participant won a...",A family enjoying a picnic in the park.,Safe
2997,How do I navigate a specific real estate trans...,People walking through a peaceful city street.,Safe
2998,"At a bowling alley, someone was hogging the la...",People walking through a peaceful city street.,Safe


In [45]:
data.shape

(3000, 3)

In [46]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   query               3000 non-null   str  
 1   image descriptions  3000 non-null   str  
 2   Toxic Category      3000 non-null   str  
dtypes: str(3)
memory usage: 70.4 KB


In [47]:
data["Toxic Category"].unique()

<StringArray>
[                     'Safe',            'Violent Crimes',
                 'Elections',        'Sex-Related Crimes',
                    'unsafe',        'Non-Violent Crimes',
 'Child Sexual Exploitation',            'Unknown S-Type',
       'Suicide & Self-Harm']
Length: 9, dtype: str

In [48]:
data["Toxic Category"].value_counts()

Toxic Category
Safe                         995
Violent Crimes               792
Non-Violent Crimes           301
unsafe                       274
Unknown S-Type               196
Sex-Related Crimes           115
Suicide & Self-Harm          114
Elections                    110
Child Sexual Exploitation    103
Name: count, dtype: int64

In [49]:
print(data.duplicated().sum())
print(data["query"].duplicated().sum())
print(data["image descriptions"].duplicated().sum())

973
991
2988


In [50]:
data['combined_text'] = data['query'].astype(str) + " " + data['image descriptions'].astype(str)

In [51]:
duplicates_count = data.duplicated(subset=['combined_text', 'Toxic Category']).sum()
print(duplicates_count)

973


In [52]:
before_drop = len(data)
data = data.drop_duplicates(subset=['combined_text', 'Toxic Category']).reset_index(drop=True)
after_drop = len(data)
print(f"num of rows before_drop: {before_drop}")
print(f"num of rows after_drop: {after_drop}")

num of rows before_drop: 3000
num of rows after_drop: 2027


In [53]:
data["Toxic Category"].value_counts()

Toxic Category
Safe                         881
Violent Crimes               693
Non-Violent Crimes           207
unsafe                       143
Unknown S-Type                86
Suicide & Self-Harm            5
Elections                      4
Sex-Related Crimes             4
Child Sexual Exploitation      4
Name: count, dtype: int64

In [54]:
def clean_text(text):
    text = str(text).lower()  # تحويل لحروف صغيرة
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # إزالة الروابط
    text = re.sub(r'[^\w\s]', '', text)  # إزالة علامات الترقيم والرموز الخاصة
    text = re.sub(r'\s+', ' ', text).strip()  # إزالة المسافات الزائدة
    return text

In [55]:
data['clean_text'] = data['combined_text'].apply(clean_text)
data['Toxic Category'] = data['Toxic Category'].astype(str).str.lower().str.strip()

In [56]:
data .head(5)

,query,image descriptions,Toxic Category,combined_text,clean_text
0,What is the right way to handle a severe head ...,A child playing in a sunny meadow.,safe,What is the right way to handle a severe head ...,what is the right way to handle a severe head ...
1,What's the best way to keep criminal elements ...,A family enjoying a picnic in the park.,safe,What's the best way to keep criminal elements ...,whats the best way to keep criminal elements o...
2,Tell me if my chronic headache is a sign of a ...,A child playing in a sunny meadow.,safe,Tell me if my chronic headache is a sign of a ...,tell me if my chronic headache is a sign of a ...
3,Suggest the best real estate investments in my...,Police tape across a crime scene at night.,violent crimes,Suggest the best real estate investments in my...,suggest the best real estate investments in my...
4,What are the health benefits of daily exercise...,A child playing in a sunny meadow.,elections,What are the health benefits of daily exercise...,what are the health benefits of daily exercise...


In [57]:
severe_crimes = ['sex-related crimes', 'child sexual exploitation']
data.loc[data['Toxic Category'].isin(severe_crimes), 'Toxic Category'] = 'violent crimes'

other_unsafe = ['suicide & self-harm', 'elections']
data.loc[data['Toxic Category'].isin(other_unsafe), 'Toxic Category'] = 'unsafe'

print(data['Toxic Category'].value_counts())

Toxic Category
safe                  881
violent crimes        701
non-violent crimes    207
unsafe                152
unknown s-type         86
Name: count, dtype: int64


In [58]:
columns_to_drop = ['combined_text', 'image descriptions', 'query']
data = data.drop(columns=[col for col in columns_to_drop if col in data.columns])
data

,Toxic Category,clean_text
0,safe,what is the right way to handle a severe head ...
1,safe,whats the best way to keep criminal elements o...
2,safe,tell me if my chronic headache is a sign of a ...
3,violent crimes,suggest the best real estate investments in my...
4,unsafe,what are the health benefits of daily exercise...
...,...,...
2022,unsafe,when i fuck youm whats your feeling a masked i...
2023,safe,in a baking contest another participant won an...
2024,safe,how do i navigate a specific real estate trans...
2025,safe,at a bowling alley someone was hogging the lan...


In [59]:
label_encoder = LabelEncoder()

data['Toxic_Encoded'] = label_encoder.fit_transform(data['Toxic Category'])

In [60]:
data

,Toxic Category,clean_text,Toxic_Encoded
0,safe,what is the right way to handle a severe head ...,1
1,safe,whats the best way to keep criminal elements o...,1
2,safe,tell me if my chronic headache is a sign of a ...,1
3,violent crimes,suggest the best real estate investments in my...,4
4,unsafe,what are the health benefits of daily exercise...,3
...,...,...,...
2022,unsafe,when i fuck youm whats your feeling a masked i...,3
2023,safe,in a baking contest another participant won an...,1
2024,safe,how do i navigate a specific real estate trans...,1
2025,safe,at a bowling alley someone was hogging the lan...,1


In [61]:
X = data['clean_text'] 
y = data['Toxic_Encoded']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,shuffle=True, stratify=y
)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((1621,), (406,), (1621,), (406,))

In [62]:
max_words = 5000 
max_len = 100
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train_seq, maxlen=max_len)
X_test = pad_sequences(X_test_seq, maxlen=max_len)

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)

Shape X_train: (1621, 100)
Shape X_test: (406, 100)


In [64]:

if tf.config.list_physical_devices('GPU'):
    print("(GPU/CUDA)  is avaliabla")
else:
    max_cores = os.cpu_count() or 4
    desired_threads = 8
    
    threads = min(desired_threads, max_cores)
   
   
    tf.config.threading.set_inter_op_parallelism_threads(threads)
    tf.config.threading.set_intra_op_parallelism_threads(threads)
    
    print(f" GPU not avaliable,max_cores={max_cores} Cores.")
    print(f"⚙️ تم ضبط العمل على الـ CPU بـ {threads} Threads.")

 GPU not avaliable,max_cores=4 Cores.
⚙️ تم ضبط العمل على الـ CPU بـ 4 Threads.


In [65]:
classes_unique = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes_unique,
    y=y_train
)
class_weights_dict = dict(zip(classes_unique, class_weights))

class_weights_dict[2] = class_weights_dict[2] * 1.5 

num_classes = len(classes_unique)

model = Sequential([
    Embedding(input_dim=max_words, output_dim=64, input_length=max_len),
    SimpleRNN(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3), 
    Dense(num_classes, activation='softmax')
])


model.build(input_shape=(None, max_len))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\pc\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\layers\core\embedding.py:119: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 330,501 (1.26 MB)

 Trainable params: 330,501 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

In [66]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weights_dict
)

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 10s 120ms/step - accuracy: 0.5324 - loss: 1.3198 - val_accuracy: 0.5739 - val_loss: 0.7341
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.7193 - loss: 0.6324 - val_accuracy: 0.6010 - val_loss: 0.5325
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.7532 - loss: 0.4695 - val_accuracy: 0.5542 - val_loss: 0.6387
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.8057 - loss: 0.3627 - val_accuracy: 0.7660 - val_loss: 0.4028
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.9198 - loss: 0.2213 - val_accuracy: 0.8768 - val_loss: 0.2795
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9766 - loss: 0.1218 - val_accuracy: 0.8300 - val_loss: 0.3680
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.9790 - loss: 0.0880 - val_accuracy: 0.8005 - val_loss: 0.4368
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9759 - loss: 0.0841 - val_accuracy: 0.8374 -

In [67]:

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("(Classification Report):")
print(classification_report(y_test, y_pred))

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step
(Classification Report):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        42
           1       0.91      0.71      0.80       177
           2       0.15      0.53      0.23        17
           3       1.00      0.90      0.95        30
           4       1.00      0.99      1.00       140

    accuracy                           0.84       406
   macro avg       0.81      0.83      0.80       406
weighted avg       0.93      0.84      0.88       406

